In [9]:
import requests
import pandas as pd

# Correct endpoint - returns all players
url = "https://fantasy.premierleague.com/api/bootstrap-static/"

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise error if status isn't 200
    data = response.json()
    
    # Players are in the 'elements' array
    players = data['elements']
    
    # Build dataframe
    player_data = []
    for player in players:
        player_data.append({
            'player_id': player['id'],
            'first_name': player['first_name'],
            'last_name': player['second_name'],
            'player_code': player['code'],
            'team_id': player['team'],
            'value': player['now_cost'] / 10,  # Convert pence to pounds
            'position': player['element_type']
        })
    
    df_players = pd.DataFrame(player_data)
    print(f"Successfully fetched {len(df_players)} players")
    print(df_players.head())
    
except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
except KeyError as e:
    print(f"Unexpected JSON structure: {e}")

Successfully fetched 555 players
   player_id first_name              last_name  player_code  team_id  value  \
0          1      David            Raya Martín       154561        1    6.0   
1          2       Kepa  Arrizabalaga Revuelta       109745        1    5.0   
2          3      Illan                Meslier       437495        1    5.0   
3          4    Gabriel   dos Santos Magalhães       226597        1    8.0   
4          5    Jurriën                 Timber       445122        1    6.5   

   position  
0         1  
1         1  
2         1  
3         2  
4         2  


In [10]:
df_players['season'] = 2627
df_players

,player_id,first_name,last_name,player_code,team_id,value,position,season
0,1,David,Raya Martín,154561,1,6.0,1,2627
1,2,Kepa,Arrizabalaga Revuelta,109745,1,5.0,1,2627
2,3,Illan,Meslier,437495,1,5.0,1,2627
3,4,Gabriel,dos Santos Magalhães,226597,1,8.0,2,2627
4,5,Jurriën,Timber,445122,1,6.5,2,2627
...,...,...,...,...,...,...,...,...
550,549,Chemsdine,Talbi,549912,20,5.5,3,2627
551,550,Djiamgone Jocelin Ta,Bi,699989,20,5.0,3,2627
552,551,Nilson,Angulo,536661,20,5.0,3,2627
553,552,Brian,Brobbey,441264,20,6.0,4,2627


In [24]:
# Make a copy to avoid modifying the original
df_new = df_players[['player_id', 'player_code', 'team_id', 'season', 'value']].copy()

# Add gameweek = 1 for all rows
df_new['gameweek'] = 1

# Get the full list of columns from the target CSV structure
target_columns = [
    'player_id', 'player_code', 'gameweek', 'fixture_id', 'name', 'web_name',
    'position_id', 'selected_by_percent', 'value', 'team_id', 'opponent_id',
    'h_a', 'kickoff_time', 'team_score', 'opponent_score', 'minutes',
    'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'yellow_cards',
    'red_cards', 'bonus', 'bps', 'total_points', 'xG', 'xA', 'xGI', 'xGC',
    'DefCon', 'ict_index', 'influence', 'creativity', 'threat', 'in_dreamteam',
    'status', 'news', 'corners_indirect_freekicks_order', 'direct_freekicks_order',
    'penalties_order', 'season'
]

# Reindex to match exact column order and add missing columns as empty/NaN
df_new = df_new.reindex(columns=target_columns)

# Fill the new columns with appropriate defaults (mostly NaN/empty)
numeric_cols = ['fixture_id', 'position_id', 'selected_by_percent', 
                'opponent_id', 'team_score', 'opponent_score', 'minutes',
                'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
                'yellow_cards', 'red_cards', 'bonus', 'bps', 'total_points',
                'xG', 'xA', 'xGI', 'xGC', 'DefCon', 'ict_index', 'influence',
                'creativity', 'threat', 'corners_indirect_freekicks_order',
                'direct_freekicks_order', 'penalties_order']

for col in numeric_cols:
    if col in df_new.columns:
        df_new[col] = 0.0 if col in ['value', 'selected_by_percent'] else 0

# String/boolean columns
df_new['name'] = ''
df_new['web_name'] = ''
df_new['h_a'] = ''
df_new['kickoff_time'] = ''
df_new['status'] = 'a'
df_new['news'] = ''
df_new['in_dreamteam'] = False

In [25]:
df_new

,player_id,player_code,gameweek,fixture_id,name,web_name,position_id,selected_by_percent,value,team_id,...,influence,creativity,threat,in_dreamteam,status,news,corners_indirect_freekicks_order,direct_freekicks_order,penalties_order,season
0,1,154561,1,0,,,0,0.0,6.0,1,...,0,0,0,False,a,,0,0,0,2627
1,2,109745,1,0,,,0,0.0,5.0,1,...,0,0,0,False,a,,0,0,0,2627
2,3,437495,1,0,,,0,0.0,5.0,1,...,0,0,0,False,a,,0,0,0,2627
3,4,226597,1,0,,,0,0.0,8.0,1,...,0,0,0,False,a,,0,0,0,2627
4,5,445122,1,0,,,0,0.0,6.5,1,...,0,0,0,False,a,,0,0,0,2627
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550,549,549912,1,0,,,0,0.0,5.5,20,...,0,0,0,False,a,,0,0,0,2627
551,550,699989,1,0,,,0,0.0,5.0,20,...,0,0,0,False,a,,0,0,0,2627
552,551,536661,1,0,,,0,0.0,5.0,20,...,0,0,0,False,a,,0,0,0,2627
553,552,441264,1,0,,,0,0.0,6.0,20,...,0,0,0,False,a,,0,0,0,2627


In [27]:
(df_new['selected_by_percent'] == 0.0).sum()

np.int64(555)

In [29]:
import os
folder = r"C:\Users\JesseOnu\fpl sql rework\gws" 
filename = os.path.join(folder, "2627gw1.csv")
df_new.to_csv(filename, index= False)
